# Maximise generalization performance on a training set

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data, target = fetch_california_housing(return_X_y=True, as_frame=True)
target *= 100  # rescale the target in k$

data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=42
)

In [2]:
data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [3]:
target.head()

0    452.6
1    358.5
2    352.1
3    341.3
4    342.2
Name: MedHouseVal, dtype: float64

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

scaler = StandardScaler()
model = make_pipeline(scaler, KNeighborsRegressor())

In [ ]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "kneighborsregressor__n_neighbors": np.logspace(0, 3, num=10).astype(
        np.int32
    ),
    "standardscaler__with_mean": [True, False],
    "standardscaler__with_std": [True, False],
} 
model_random_search = RandomizedSearchCV(
    model,
    param_distributions=param_distributions,
    n_iter = 20,
    n_jobs = 2,
    verbose = 1,
    random_state = 1,
)

model_random_search.fit(data_train, target_train)
model_random_search.best_params__

In [14]:
import pandas as pd

cv_results = pd.DataFrame(model_random_search.cv_results_)

In [15]:
column_name_mapping = {
    "param_kneighborsregressor__n_neighbors": "n_neighbors",
    "param_standardscaler__with_mean": "centering",
    "param_standardscaler__with_std": "scaling",
    "mean_test_score": "mean test score",
}

cv_results = cv_results.rename(columns=column_name_mapping)
cv_results = cv_results[column_name_mapping.values()].sort_values(
    "mean test score", ascending=False
)

In [16]:
column_scaler = ["centering", "scaling"]
cv_results[column_scaler] = cv_results[column_scaler].astype(np.int64)
cv_results["n_neighbors"] = cv_results["n_neighbors"].astype(np.int64)
cv_results

,n_neighbors,centering,scaling,mean test score
17,10,0,1,0.687926
18,4,0,1,0.674812
6,46,0,1,0.668778
9,100,0,1,0.648317
16,2,1,1,0.629772
15,215,1,1,0.617295
12,215,0,1,0.617295
10,464,1,1,0.567164
0,1,0,1,0.508809
13,1000,1,1,0.486503


Selecting the best performing models (i.e. above R2 score of ~0.68), we observe that in this case:

- scaling the data is important. All the best performing models use scaled features;

- centering the data does not have a strong impact. Both approaches, centering and not centering, can lead to good models;

- using some neighbors is fine but using too many is a problem. In particular no pipeline with n_neighbors=1 can be found among the best models. However, scaling features has an even stronger impact than the choice of n_neighbors in this problem